In [6]:
import pandas as pd

df = pd.read_csv(r"C:\Users\angel\DS8_SIA\DS8_SIA_Project\project\01_data\processed\final_priority_geo.csv")

In [7]:
df.head()

,SQLDATE,EventCode,GoldsteinScale,NumMentions,AvgTone,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL,score_mentions,score_goldstein,score_tone,priority_score,geo_level
0,2022-08-07,190,-10.0,1420,-4.161978,4,39.9289,116.388,https://www.sfgate.com/news/article/China-keep...,0.968377,1.000000,0.566864,0.851085,Level2
1,2025-02-27,193,-10.0,1026,-3.964070,4,39.9289,116.388,https://www.ksat.com/news/world/2025/02/27/tai...,0.691497,1.000000,0.539704,0.676809,Level2
2,2021-01-19,190,-10.0,1406,-1.271789,4,24.0000,119.000,https://www.9news.com.au/world/taiwan-stages-m...,0.958538,1.000000,0.000000,0.675123,Level2
3,2021-10-03,192,-9.5,840,-5.283206,4,39.9289,116.388,https://www.mdjonline.com/tribune/china-contin...,0.560787,0.821429,0.720740,0.634837,Level2
4,2020-12-31,151,-7.2,1190,-3.208156,4,24.0000,119.000,https://www.sfgate.com/news/article/China-accu...,0.806746,0.000000,0.435963,0.614837,Level2


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 169 entries, 0 to 168
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   SQLDATE          169 non-null    datetime64[ns]
 1   EventCode        169 non-null    int64         
 2   GoldsteinScale   169 non-null    float64       
 3   NumMentions      169 non-null    int64         
 4   AvgTone          169 non-null    float64       
 5   ActionGeo_Type   169 non-null    int64         
 6   ActionGeo_Lat    169 non-null    float64       
 7   ActionGeo_Long   169 non-null    float64       
 8   SOURCEURL        169 non-null    object        
 9   score_mentions   169 non-null    float64       
 10  score_goldstein  169 non-null    float64       
 11  score_tone       169 non-null    float64       
 12  priority_score   169 non-null    float64       
 13  geo_level        169 non-null    object        
dtypes: datetime64[ns](1), float64(8), int64(3)

In [8]:
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'])

In [9]:
latest_date = df['SQLDATE'].max()

df_recent = df[
    df['SQLDATE'] >= latest_date - pd.Timedelta(days=14)
]

In [10]:
df_recent = df_recent[
    df_recent['geo_level'].isin(['Level1', 'Level2'])
]

In [11]:
coords = df_recent[
    ['ActionGeo_Lat', 'ActionGeo_Long']
].values

In [28]:
from sklearn.cluster import DBSCAN
import numpy as np

coords_rad = np.radians(coords)

In [29]:
kms_per_radian = 6371.0088

epsilon = 5 / kms_per_radian

In [30]:
db = DBSCAN(
    eps=epsilon,
    min_samples=3,
    algorithm='ball_tree',
    metric='haversine'
)

clusters = db.fit_predict(coords_rad)

In [31]:
df_recent['cluster'] = clusters

In [32]:
import folium

m = folium.Map(
    location=[24.5, 120.5],
    zoom_start=6,
    tiles='CartoDB positron'
)

In [33]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

unique_clusters = df_recent['cluster'].unique()

colors = plt.cm.tab10(range(len(unique_clusters)))

cluster_color_map = {
    cluster: mcolors.to_hex(colors[i])
    for i, cluster in enumerate(unique_clusters)
}

In [34]:
for _, row in df_recent.iterrows():

    cluster = row['cluster']

    # noise는 회색
    if cluster == -1:
        color = 'gray'
    else:
        color = cluster_color_map[cluster]

    folium.CircleMarker(
        location=[
            row['ActionGeo_Lat'],
            row['ActionGeo_Long']
        ],
        radius=6,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=f"""
        Cluster: {cluster}<br>
        Score: {row['priority_score']:.3f}<br>
        Date: {row['SQLDATE']}
        """
    ).add_to(m)

In [35]:
m.save("dbscan_map.html")

In [38]:
df[
    ['ActionGeo_Lat', 'ActionGeo_Long']
].drop_duplicates().shape

(20, 2)

In [36]:
df_recent[
    ['ActionGeo_Lat', 'ActionGeo_Long']
].drop_duplicates().shape

(2, 2)

In [ ]:
df_recent.groupby(
    ['ActionGeo_Lat', 'ActionGeo_Long']
).size().sort_values(ascending=False).head(20)

ActionGeo_Lat  ActionGeo_Long
25.0478        121.532           3
39.9289        116.388           3
dtype: int64

In [39]:
df.groupby(
    ['ActionGeo_Lat', 'ActionGeo_Long']
).size().sort_values(ascending=False).head(20)

ActionGeo_Lat  ActionGeo_Long
39.9289        116.388           79
25.0478        121.532           41
24.0000        119.000           19
24.4367        118.318            5
24.1098        113.005            3
24.0000        121.000            3
21.4167        121.500            3
29.4639        112.615            2
32.5261        119.739            2
22.4956        120.614            2
20.6827        109.951            1
22.1094        120.874            1
22.4178        120.549            1
25.0327        121.275            1
26.5450        117.843            1
28.7925        117.262            1
32.9500        116.642            1
37.5664        127.000            1
41.0000        123.000            1
42.8333        124.633            1
dtype: int64